# Lab 05 — A/B Test Analysis — Old vs New Page
**Experimentation Basics Track** · Intermediate · ~50 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Compute conversion rates by experiment group
2. Run a two-proportion z-test and interpret p-value
3. Build a 95% CI for the conversion difference
4. Estimate MDE at 80% power and make a ship decision

## Datasets (this folder)
- `ab_test.csv` — auto-download from `https://raw.githubusercontent.com/TimileyinSamuel/A-B-Testing-for-E-Commerce-Website/main/ab_test.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-05-ab-test-analysis/lab-05-ab-test-analysis.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`ab_test.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-05-ab-test-analysis"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/labs/lab-05-ab-test-analysis/bundle/dataset.zip"
NEED = ["ab_test.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Experimentation Basics Track: Rates, Z-Test, Ship/No-Ship

> **Scenario:** `ab_test.csv` logs **294,478 sessions** with `con_treat` (control/treatment), `page`, `converted` (0/1). Compute conversion rates, a two-proportion z-test, a 95% CI, and a minimum detectable effect — then **conclude whether to ship**.
>
> **You will learn:** group-by rates, standard error, z-test (manual or scipy), power sketch, peeking pitfall.
> **Time:** ~50 minutes. **Level:** Intermediate. **Needs:** pandas + math/scipy. **Env:** 🟢 Colab only.

The whole lab hangs on one decision: *does the new page convert better than the old one, and is the evidence strong enough to ship?* Everything else — rates, z, CI, MDE — is scaffolding for that sentence. You will work from raw session rows to a defensible **do not ship / ship** conclusion, and you will quantify not only the observed gap but also the smallest gap this sample could have detected reliably. Keep the decision question in view through every step: any number that does not feed the ship call is decoration.

### Experiment mental map

| Concept | Formula / code | Meaning |
|---|---|---|
| Conversion rate | `conversions / n` | p̂ per group |
| Pooled SE | `√(p(1-p)(1/n₁+1/n₀))` | null variance for z |
| z-stat | `(p₁ − p₀) / SE` | how extreme is the gap |
| 95% CI (diff) | `(p̂₁−p̂₀) ± 1.96·SE_unpooled` | plausible range for lift |
| MDE @ 80% power | `(1.96+0.84)·SE₀` | smallest gap you can reliably see |

Each row of the table is one rung on the analysis ladder. The conversion rate is the raw fact about each group — how many sessions converted, out of how many. The pooled standard error answers a different question: *if the two groups truly had the same conversion rate*, how much would the gap between samples jitter by chance alone? The z-statistic puts your observed gap on that chance scale, and the two-sided p-value turns z into “how often would noise alone look this extreme?” The confidence interval then translates the same evidence back into effect-size units: a range of plausible true lift values, which is the language a ship decision actually needs. Finally, MDE looks forward rather than backward — with these sample sizes, how big would a real effect have to be before you could reliably see it? The numbered sections walk this ladder in order: load and QA the data, compute rates, run the z-test, build the CI, read off the MDE, and write the decision.

A note on formulas above: they are written in plain Unicode (p̂, √, ±, subscripts) so they render anywhere — no MathJax required. When you type the same expressions in code, `math.sqrt` replaces √, `1.96 * se` replaces 1.96·SE, and so on; the mapping is one-to-one.

---

### 1. Load data (local first, Colab fallback)

**Why:** A rate means nothing until you know *n* and that the group labels actually match the pages users saw. This step loads the 294k session log, shows the raw group sizes, and forces the classic experiment QA check: cross-tab `group × page` before any hypothesis test. Logging bugs and bot traffic hide in mismatched rows, and fixing them after you have seen p-values is how teams ship on contaminated data.

In [ ]:
import math, os
import pandas as pd
from scipy import stats

def load_ab():
    local = "ab_test.csv"
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/TimileyinSamuel/A-B-Testing-for-E-Commerce-Website/main/ab_test.csv",
            local,
        )
    return pd.read_csv(local)

df = load_ab()
print(df.shape)          # (294478, 5)
print(df.head(3))
print(df["con_treat"].value_counts())
# control     147202
# treatment   147276
print(df["page"].value_counts())
# old_page / new_page each 147239 (note: page and group are almost aligned —
# a few mismatched rows exist in this dataset; classic QA check)


> Always cross-tab `group × page`. Mismatches mean logging bugs or bot traffic — fix before testing.

In [ ]:
print(pd.crosstab(df["con_treat"], df["page"]))


**What to notice:**
- The table has **294,478** rows — session-level, one row per assignment plus conversion outcome.
- Group sizes are nearly balanced: control **147,202** vs treatment **147,276** sessions.
- Page labels are also nearly balanced (old/new ≈ **147,239** each), but they are *not* perfectly identical to the group counts — a few mismatched rows exist in this dataset.
- Those mismatch rows are the classic QA signal: an assigned group should see only its page; disagreement means logging inconsistency worth flagging before you trust the test.

> **Pitfall:** Never skip the crosstab. If `con_treat` and `page` disagree, your “treatment effect” may be measuring a logging bug. Decide a policy (drop mismatches, or treat assignment as source of truth), apply it *before* computing rates, and say so in the write-up.

---

### 2. Conversion rates by group

**Why:** The rates are the entire observed effect — every later statistic (z, CI, MDE) is a function of these two proportions and their sample sizes. Computing them first keeps the ship decision anchored in something a product manager can read directly: “control converted at 12.04%, treatment at 11.89%.” If the direction of the gap surprises you here, you already know the test will not support a celebratory ship note.

In [ ]:
rates = df.groupby("con_treat")["converted"].agg(["sum", "count", "mean"])
rates.columns = ["conversions", "n", "rate"]
print(rates.round(6))


Expected:

| group | conversions | n | rate |
|---|---|---|---|
| control | 17723 | 147202 | **0.120399** |
| treatment | 17514 | 147276 | **0.118920** |

**What to notice:**
- Control rate **0.120399** (17,723 / 147,202) vs treatment rate **0.118920** (17,514 / 147,276).
- Treatment is **lower**, not higher: raw diff ≈ **−0.00148** (−1.23% relative).
- The absolute gap is tiny — about 1.5 fewer conversions per 1,000 sessions — so whether it is real or noise is exactly what the z-test must answer.
- `sum` and `count` are kept alongside `mean` because the z-test needs integer conversion totals, not just proportions.

> **Pitfall:** A negative point estimate does not by itself mean the new page is worse — sampling noise can flip small gaps. Equally, do not call it “flat” until the CI in step 4 says zero is plausible. Report the direction, then let the test quantify your uncertainty.

---

### 3. Two-proportion z-test

**Why:** The rates differ, but by how much *relative to chance*? The two-proportion z-test formalizes that question: pool both samples under the null hypothesis (equal conversion), compute the standard error of the difference, and see how many of those standard errors your observed gap occupies. This is the first number a stats-literate reviewer will ask for, and it feeds the ship/no-ship call directly at α = 0.05.

Manual (no scipy required):

In [ ]:
c0, n0 = 17723, 147202   # control
c1, n1 = 17514, 147276   # treatment
p0, p1 = c0 / n0, c1 / n1
p_pool = (c0 + c1) / (n0 + n1)
se_pool = math.sqrt(p_pool * (1 - p_pool) * (1/n0 + 1/n1))
z = (p1 - p0) / se_pool
p_value = 2 * (1 - stats.norm.cdf(abs(z)))
print(f"p0={p0:.6f}  p1={p1:.6f}  diff={p1-p0:.6f}")
print(f"z={z:.4f}  p={p_value:.4f}")
# p0=0.120399  p1=0.118920  diff=-0.001480
# z=-1.2369  p=0.2161


Equivalent scipy:

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
# optional: counts = [c1, c0]; nobs = [n1, n0]; z, p = proportions_ztest(counts, nobs)


**What to notice:**
- Pooled rate sits between the two group rates; under H₀ both groups share this single p.
- **z ≈ −1.2369** — the observed gap is only about 1.2 null standard errors from zero, well inside ordinary noise.
- **p ≈ 0.2161** — a gap at least this large would show up roughly 1 in 5 times from pure chance if the pages were identical.
- The sign of z is negative because treatment was computed first in the numerator; magnitude and p are what matter for the decision.

**Decision at α = 0.05:** p ≈ 0.216 > 0.05 → **fail to reject** H₀. No significant difference.

> **Pitfall:** “Fail to reject” is not “the pages are identical” and it is not “the new page is fine.” It means *this sample did not produce enough evidence to distinguish the pages*. The CI next makes that ambiguity quantitative instead of binary.

---

### 4. 95% CI for the difference

**Why:** A p-value answers “is it distinguishable from zero?” but a ship decision needs “what magnitudes are still plausible?” The confidence interval puts the lift back into percentage-point units, so stakeholders can see the full range of true effects consistent with the data — including zero, and including modest harm or modest gain. If zero is inside the interval, the honest recommendation is not to ship on this evidence alone.

In [ ]:
se_unpooled = math.sqrt(p1*(1-p1)/n1 + p0*(1-p0)/n0)
diff = p1 - p0
ci = (diff - 1.96*se_unpooled, diff + 1.96*se_unpooled)
print(f"diff={diff:.6f}  95% CI=({ci[0]:.6f}, {ci[1]:.6f})")
# diff=-0.001480  95% CI=(-0.003824, 0.000865)


**What to notice:**
- Point estimate **diff ≈ −0.001480** (treatment minus control).
- 95% CI **(−0.003824, 0.000865)** spans zero — plausible true effects run from a small loss to a small gain.
- The unpooled SE is used for the CI (each group keeps its own variance); the pooled SE was for the null hypothesis test — different tools, both correct for their job.
- In practical terms the interval allows up to roughly **0.38 pp worse** or **0.09 pp better**, none of which clears a business bar on its own.

CI **contains 0** → ship decision: **do not ship** the new page on this evidence (and the point estimate is slightly worse).

> **Pitfall:** Do not read the CI as “there is a 95% probability the true lift is in this interval” in frequentist terms — say instead that 95% of intervals constructed this way would cover the true lift. For the memo, the operative fact is simpler: zero is inside, so the data do not support a ship.

---

### 5. Minimum detectable effect @ 80% power

**Why:** “Not significant” can mean “no effect” or “we never had the resolution to see a modest effect.” MDE separates those stories: with these sample sizes, how big would a real conversion lift have to be before this test would reliably flag it? That number sets expectations for the next experiment — if the business cares about 0.1 pp but you can only detect 0.34 pp, you need more traffic, a sharper metric, or a longer run *before* launch, not after.

In [ ]:
# alpha=0.05 two-sided, power=0.80 → z_a + z_b ≈ 1.96 + 0.84
mde = (1.959964 + 0.841621) * se_pool
print(f"MDE absolute: {mde:.6f}  (~{mde/p0*100:.2f}% relative to control)")
# MDE absolute: 0.003351  (~2.78% relative to control)


**What to notice:**
- **MDE ≈ 0.003351** absolute — about **0.34 percentage points** of conversion.
- Relative to the control rate that is ≈ **2.78%** — the smallest lift this design catches with 80% power at α = 0.05.
- The observed |diff| ≈ 0.00148 is *below* the MDE, which is exactly why the test came back non-significant: the experiment was underpowered for an effect this small.
- Read the MDE before the p-value when planning the *next* test: choose n from the business-relevant effect, not from whatever traffic happens to arrive.

With these sample sizes you can only reliably detect lifts ≳ 0.34 percentage points.

> **Pitfall:** Checking the z-test every hour and stopping when p < 0.05 inflates false positives — each peek is another chance to cross the threshold by noise, so your real α is far above 0.05. Pre-register sample size / duration from the MDE; use sequential methods (e.g. mSPRT, group-sequential boundaries) if you must peek. Peeking is the fastest way to “discover” effects that do not replicate.

---

## Exercises (do these!)

### Exercise 1 — Conversion by group
Print `conversions`, `n`, and `rate` for control and treatment (6 d.p.).
*Expected: control 17723/147202 = 0.120399 · treatment 17514/147276 = 0.118920.*

**Follow-up:** Express the lift relative to control. Check: about minus 1.23 percent.

<details>
<summary>Hint</summary>

`df.groupby("con_treat")["converted"].agg(["sum","count","mean"])`.
</details>

### Exercise 2 — Z-test + p-value
Run the two-proportion z-test. Report z (4 d.p.) and p (4 d.p.). Significant at α=0.05?
*Expected: z ≈ −1.2369, p ≈ 0.2161 → not significant.*

**Follow-up:** Confirm the observed diff lies inside the 95 percent CI. Check: minus 0.003824 < diff < 0.000865.

<details>
<summary>Hint</summary>

Pool proportions for SE under H₀; two-sided p via `2*(1-Φ(|z|))`.
</details>

### Exercise 3 — MDE at 80% power
Using the pooled SE from the null, compute MDE = (1.96+0.84)·SE. Report absolute and % of control rate.
*Expected: ≈ 0.00335 absolute ≈ 2.78% of 0.1204.*

**Follow-up:** Is the observed effect even detectable — compare its size to the MDE. Check: no, the test is underpowered for this effect.

<details>
<summary>Hint</summary>

`mde = (1.96 + 0.84) * se_pool`; relative = `mde / p0`.
</details>

---

## Solutions

In [ ]:
# --- Solution 1 ---
rates = df.groupby("con_treat")["converted"].agg(["sum", "count", "mean"])
print(rates)
# control:   17723 / 147202 = 0.120399
# treatment: 17514 / 147276 = 0.118920

# --- Solution 2 ---
c0, n0 = 17723, 147202
c1, n1 = 17514, 147276
p0, p1 = c0/n0, c1/n1
p = (c0+c1)/(n0+n1)
se = math.sqrt(p*(1-p)*(1/n0 + 1/n1))
z = (p1-p0)/se
pv = 2*(1 - stats.norm.cdf(abs(z)))
print(f"z={z:.4f} p={pv:.4f}")   # z=-1.2369 p=0.2161
print("significant?", pv < 0.05)  # False

# --- Solution 3 ---
mde = (1.959964 + 0.841621) * se
print(f"MDE={mde:.6f} ({mde/p0*100:.2f}% of control)")
# MDE=0.003351 (2.78% of control)

# --- Follow-up 1 ---
rel = round((p1 - p0) / p0, 4)
print(rel)  # ~-0.0123
assert rel == -0.0123

# --- Follow-up 2 ---
se_u = math.sqrt(p0 * (1 - p0) / n0 + p1 * (1 - p1) / n1)
lo, hi = (p1 - p0) - 1.96 * se_u, (p1 - p0) + 1.96 * se_u
print(round(lo, 6), round(hi, 6))
assert lo < (p1 - p0) < hi
assert abs(lo + 0.003824) < 1e-5 and abs(hi - 0.000865) < 1e-5

# --- Follow-up 3 ---
print(abs(p1 - p0) < mde)  # True -> underpowered for this effect
assert abs(p1 - p0) < mde


### What to learn next
- Chi-square test of independence on the same 2×2 table (should agree with z).
- CUPED / variance reduction; sequential testing (mSPRT).
- Power *before* launch: choose n from business MDE, not after the fact.
- Cheat sheet: rates → z/p → CI → MDE → decision; never peek-and-stop naïvely.

*Files in this folder: `ab_test.csv` (local; raw GitHub URL in `load_ab()`). Paste any block into Python/Jupyter and run top-to-bottom.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
